# Setup

In [1]:
%load_ext autoreload
%autoreload 2
import os
import sys
import pandas as pd
from dotenv import load_dotenv
from torch.backends import cudnn

# enforce more deterministic behavior
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
cudnn.deterministic = True
cudnn.benchmark = False

sys.path.append("..")
load_dotenv("../../.env")

from pneuma_seeker.core.materializer.operation.semantic_join import SemanticJoiner
from pneuma_seeker.model.interface.model_factory import get_embed_model

In [2]:
embed_model_path = "model/weight/bge-base"
embed_model = get_embed_model()(embed_model_path)

In [3]:
left_df = pd.DataFrame([
    {"name": "MIT", "city": "Cambridge", "state": "MA", "country": "USA",
     "founded": 1861, "students": 11466, "endowment_billion": 23.5,
     "type": "Private", "colors": "Cardinal Red, Silver Gray", "nickname": "Engineers"},

    {"name": "UChicago", "city": "Chicago", "state": "IL", "country": "USA",
     "founded": 1890, "students": 17934, "endowment_billion": 10.3,
     "type": "Private", "colors": "Maroon", "nickname": "Maroons"},

    {"name": "Stanford", "city": "Stanford", "state": "CA", "country": "USA",
     "founded": 1885, "students": 16164, "endowment_billion": 37.8,
     "type": "Private", "colors": "Cardinal", "nickname": "Cardinal"},

    {"name": "Cal Tech", "city": "Pasadena", "state": "CA", "country": "USA",
     "founded": 1891, "students": 2237, "endowment_billion": 4.1,
     "type": "Private", "colors": "Orange, White", "nickname": "Beavers"},

    {"name": "Harvard Univ", "city": "Cambridge", "state": "MA", "country": "USA",
     "founded": 1636, "students": 21900, "endowment_billion": 53.2,
     "type": "Private", "colors": "Crimson", "nickname": "Crimson"},

    {"name": "WashU", "city": "St. Louis", "state": "MO", "country": "USA",
     "founded": 1853, "students": 16900, "endowment_billion": 15.3,
     "type": "Private", "colors": "Red, Green", "nickname": "Bears"},

    {"name": "UC Berkeley", "city": "Berkeley", "state": "CA", "country": "USA",
     "founded": 1868, "students": 45700, "endowment_billion": 6.9,
     "type": "Public", "colors": "Blue, Gold", "nickname": "Golden Bears"},

    {"name": "Princeton", "city": "Princeton", "state": "NJ", "country": "USA",
     "founded": 1746, "students": 8500, "endowment_billion": 37.7,
     "type": "Private", "colors": "Orange, Black", "nickname": "Tigers"},

    {"name": "Yale", "city": "New Haven", "state": "CT", "country": "USA",
     "founded": 1701, "students": 14100, "endowment_billion": 42.3,
     "type": "Private", "colors": "Yale Blue", "nickname": "Bulldogs"},

    {"name": "University of Washington", "city": "Seattle", "state": "WA", "country": "USA",
     "founded": 1861, "students": 49300, "endowment_billion": 4.9,
     "type": "Public", "colors": "Purple, Gold", "nickname": "Huskies"}
])
right_df = pd.DataFrame([
    {"university": "Massachusetts Institute of Technology", "location_city": "Cambridge",
     "location_state": "Massachusetts", "nation": "United States",
     "year_established": 1861, "enrollment": 11500, "fund_billion": 23.7,
     "control": "Private research", "school_colors": "Red and Gray", "sports_name": "MIT Engineers"},

    {"university": "University of Chicago", "location_city": "Chicago",
     "location_state": "Illinois", "nation": "US",
     "year_established": 1890, "enrollment": 18000, "fund_billion": 10.2,
     "control": "Private research university", "school_colors": "Maroon", "sports_name": "Chicago Maroons"},

    {"university": "Leland Stanford Junior University", "location_city": "Stanford",
     "location_state": "California", "nation": "U.S.",
     "year_established": 1885, "enrollment": 16200, "fund_billion": 37.7,
     "control": "Private research", "school_colors": "Cardinal", "sports_name": "Stanford Cardinal"},

    {"university": "California Institute of Technology", "location_city": "Pasadena",
     "location_state": "California", "nation": "United States",
     "year_established": 1891, "enrollment": 2200, "fund_billion": 4.2,
     "control": "Private research", "school_colors": "Orange and White", "sports_name": "Beavers"},

    {"university": "Harvard University", "location_city": "Cambridge",
     "location_state": "Massachusetts", "nation": "USA",
     "year_established": 1636, "enrollment": 22000, "fund_billion": 53.3,
     "control": "Private Ivy League", "school_colors": "Crimson", "sports_name": "Crimson"},

    {"university": "Washington University in St. Louis", "location_city": "St. Louis",
     "location_state": "Missouri", "nation": "United States",
     "year_established": 1853, "enrollment": 17000, "fund_billion": 15.2,
     "control": "Private research", "school_colors": "Red and Green", "sports_name": "Bears"},

    {"university": "University of California, Berkeley", "location_city": "Berkeley",
     "location_state": "California", "nation": "US",
     "year_established": 1868, "enrollment": 46000, "fund_billion": 6.8,
     "control": "Public research", "school_colors": "Blue and Gold", "sports_name": "Golden Bears"},

    {"university": "Princeton University", "location_city": "Princeton",
     "location_state": "New Jersey", "nation": "USA",
     "year_established": 1746, "enrollment": 8600, "fund_billion": 37.6,
     "control": "Private Ivy League", "school_colors": "Orange and Black", "sports_name": "Tigers"},

    {"university": "Yale University", "location_city": "New Haven",
     "location_state": "Connecticut", "nation": "US",
     "year_established": 1701, "enrollment": 14000, "fund_billion": 42.4,
     "control": "Private Ivy League", "school_colors": "Blue", "sports_name": "Bulldogs"},

    {"university": "Washington State University", "location_city": "Pullman",
     "location_state": "Washington", "nation": "United States",
     "year_established": 1890, "enrollment": 31000, "fund_billion": 1.1,
     "control": "Public research", "school_colors": "Crimson and Gray", "sports_name": "Cougars"}
])

In [4]:
semantic_joiner = SemanticJoiner(embed_model)
res = semantic_joiner.semantic_join(
    left_df,
    right_df,
    ["name", "city", "state"],
    ["university", "location_city", "location_state"],
    alpha=0.6,
    threshold=0.6
)

Embedding left (concat):   0%|          | 0/1 [00:00<?, ?it/s]

Embedding right (concat):   0%|          | 0/1 [00:00<?, ?it/s]

Edit similarity (concat):   0%|          | 0/10 [00:00<?, ?it/s]

Materializing joined rows:   0%|          | 0/28 [00:00<?, ?it/s]

In [5]:
res.to_csv("some2.csv", index=False)